# PLAID iDOT Pipeline

## Overview
This notebook runs the complete PLAID iDOT pipeline, which converts compound screening layouts into dispense protocols for the iDOT liquid handling platform.

## What it does:
1. **Loads configuration** from `config/config.json` (user name, protocol name, input files)
2. **Reads compound information** and screening layouts
3. **Performs intelligent stock selection** - finds the best matching stock for each compound
4. **Normalizes volumes** - applies solvent-family normalization while respecting configured solvent percentage caps
5. **Generates iDOT protocol** - creates the dispense instructions CSV file
6. **Creates liquids mapping** - identifies unique compounds and assigns them to source plate wells
7. **Calculates source plate prep** (optional) - determines how much of each stock needs to be prepared in the source plate

## Output files:
- `{user_name}_{protocol_name}_protocol_{timestamp}.csv` - iDOT dispense protocol
- `{user_name}_{protocol_name}_liquids_{timestamp}.csv` - Liquids mapping with source wells
- `{user_name}_{protocol_name}_source_plate_prep_{timestamp}.txt` - Source plate preparation instructions (if enabled)

## To run:
Simply execute the cell below. All configuration is in `config/config.json`.

In [1]:
import sys
from pathlib import Path

# Ensure project root is in path for imports
project_root = Path.cwd()
if not (project_root / "config" / "config.json").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.iplaid.pipeline import run_pipeline

print("=" * 80)
print("PLAID iDOT PIPELINE EXECUTION")
print("=" * 80)
print(f"Project root: {project_root}\n")

# Run the complete pipeline
result = run_pipeline(project_root=project_root, include_source_prep=True)

# Display summary
print("\n" + "=" * 80)
print("PIPELINE EXECUTION COMPLETE")
print("=" * 80)
print(f"\n📊 Results Summary:")
print(f"  • Input compounds: {len(result['df'])}")
print(f"  • Dispense rows: {len(result['all_rows'])}")
print(f"  • Unique liquids: {len(result['liquid_table_export'])}")
print(f"  • Solvent families: {len(result['solvent_summary'])}")
print(f"  • Max DMSO: {result['max_dmso_ul']:.1f} µL")

print(f"\n📁 Output files generated:")
print(f"  • Protocol: {result['paths']['out_idot'].name}")
print(f"  • Liquids: {result['paths']['out_liquids'].name}")
if result.get('source_prep_volumes'):
    print(f"  • Source plate prep instructions generated")

print(f"\n✅ Pipeline execution successful!")


PLAID iDOT PIPELINE EXECUTION
Project root: /Users/takar834/Documents/UU/TIMED/Tools/PLAID_iDOT


PRE-FLIGHT VALIDATION: Concentration Feasibility Analysis

Configuration: max_dmso_pct = 0.1%
Analysis: 13 unique compound-concentration pairs
  ✓ Feasible with current config: 13/13

RESULT: ✅ ALL CONCENTRATIONS FEASIBLE

✓ All 13 concentrations achievable
✓ Current DMSO limit (0.1%) is sufficient
✓ Pipeline can proceed


✓ STOCK CONCENTRATION ASSIGNMENT COMPLETED
Input unique (compound, target_conc) pairs: 13
Unique (compound, stock) assignments created: 9

Detailed assignments:

  DMSO:
    • Target      0.0 µM  →  Stock(s): [np.float64(0.0)] (21 wells)

  Dasatinib:
    • Target      0.1 µM  →  Stock(s): [np.float64(0.1)] (7 wells)
    • Target      0.3 µM  →  Stock(s): [np.float64(1.0)] (7 wells)
    • Target      1.0 µM  →  Stock(s): [np.float64(1.0)] (7 wells)

  Etoposide:
    • Target      3.0 µM  →  Stock(s): [np.float64(10.0)] (7 wells)
    • Target     10.0 µM  →  Stock(s): [np